In [1]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
df = pd.read_csv("F:/X ray dataset/Second Version/Data_Entry_2017.csv")
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [7]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class ChestXrayDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

        self.labels = df['Finding Labels'].values

        # all possible classes in NIH dataset
        self.classes = [
            "Atelectasis", "Consolidation", "Infiltration", "Pneumothorax",
            "Edema", "Emphysema", "Fibrosis", "Effusion",
            "Pneumonia", "Pleural_Thickening", "Cardiomegaly",
            "Nodule", "Mass", "Hernia"
        ]

    def encode_labels(self, label_str):
        label_list = label_str.split('|')
        target = torch.zeros(len(self.classes))

        for i, c in enumerate(self.classes):
            if c in label_list:
                target[i] = 1.0

        return target

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = f"{self.image_dir}/{row['Image Index']}"
        image = Image.open(img_path).convert("RGB")

        label = self.encode_labels(row['Finding Labels'])

        if self.transform:
            image = self.transform(image)

        return image, label

In [9]:
import sys
sys.path.append('D:/cxr-triage')
from src.data.transforms import get_train_transforms
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("F:/X ray dataset/Second Version/images_001/images/00000001_000.png").convert('RGB')
transform = get_train_transforms(image_size=224)  # no use_clahe passed -> should default False
img_t = transform(img)

print("Shape:", img_t.shape)  # should be [3, 224, 224]
print("Min/Max:", img_t.min().item(), img_t.max().item())

Shape: torch.Size([3, 224, 224])
Min/Max: -1.8952821493148804 1.8208280801773071
